In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import pandas as pd
import numpy as np

from src.evaluation import (
    semantic_consistency,
    retrieval_consistency,
)

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"

INPUT_PATH = DATA_DIR / "case_embeddings.parquet"

ANONYMIZED_DIR = DATA_DIR / "anonymized_outputs"

OUTPUT_DIR = DATA_DIR / "results"
OUTPUT_DIR.mkdir(exist_ok=True)

CLUSTER_K = [5, 10, 25, 50, 100]

NEIGHBOR_K = [100, 200, 300, 400, 500]

In [ ]:
df = pd.read_parquet(INPUT_PATH)
X = df.to_numpy(dtype=float)
print(f"Loaded {X.shape[0]} embeddings.")

In [ ]:
results = []

algorithms = {
    "MDAV-C": "mdav_c_k{}.parquet",
    "RMDAV-M": "rmdav_m_k{}.parquet",
}

for algorithm, pattern in algorithms.items():

    for cluster_k in CLUSTER_K:

        print(f"{algorithm} | k={cluster_k}")

        df_anon = pd.read_parquet(
            ANONYMIZED_DIR / pattern.format(cluster_k)
        )

        X_anon = df_anon.to_numpy(dtype=float)

        sc = semantic_consistency(
            X,
            X_anon,
        )

        for neighbor_k in NEIGHBOR_K:

            rc = retrieval_consistency(
                X,
                X_anon,
                k=neighbor_k,
            )

            results.append(
                {
                    "algorithm": algorithm,
                    "cluster_k": cluster_k,
                    "neighbor_k": neighbor_k,
                    "semantic_consistency": sc,
                    "retrieval_consistency": rc,
                }
            )

In [ ]:
results_df = pd.DataFrame(results)

results_df

In [ ]:
results_df.to_csv(
    OUTPUT_DIR / "utility_results.csv",
    index=False,
)

print("Utility evaluation completed successfully.")